In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from anndata import AnnData
from typing import Tuple, Literal
from scipy.stats import pearsonr

In [ ]:
for mod in [np, pd, sc, gp]:
    print(mod, mod.__version__)

In [ ]:
def vprint(msg, verbose):
    if verbose:
        print(msg)

def get_genesets(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype_signature: str,
    direction: str,
    threshold_pval: float,
    threshold_lfc: float, 
) -> Tuple[dict[str, list[str]], pd.DataFrame]:

    # construct signature gene set
    select = ((df_hcd_all.adj_p_value <= threshold_pval) & (df_hcd_all.celltype == celltype_signature))
    if direction == "upregulated":
        select = select & (df_hcd_all.log_fc >= threshold_lfc)
    elif direction == "downregulated":
        select = select & (df_hcd_all.log_fc <= threshold_lfc)
    elif direction == "both":
        select = select & (df_hcd_all.log_fc.abs() >= threshold_lfc)         
    
    df_hcd_ct = df_hcd_all.loc[select]
    
    gene_set_dict = {}
    gene_set_df = pd.DataFrame()
    for cytokine_i, cytokine in enumerate(df_hcd_ct.cytokine.unique()):
        gene_set = df_hcd_ct.loc[df_hcd_ct.cytokine == cytokine].gene.values
        gene_set_shared = np.intersect1d(gene_set, adata.var_names)
        gene_set_df.loc[cytokine_i, "cytokine"] = cytokine
        gene_set_df.loc[cytokine_i, "num_genes_signature"] = len(gene_set)
        gene_set_df.loc[cytokine_i, "num_shared_genes_signature"] = len(gene_set_shared)
        gene_set_df.loc[cytokine_i, "frac_shared_genes_signature"] = len(gene_set_shared) / len(gene_set)
        gene_set_dict[cytokine] = gene_set_shared        

    return gene_set_dict, gene_set_df


def compute_mu_and_sigma(adata: AnnData, contrast_column: str, condition: str) -> pd.DataFrame:

    group = adata[adata.obs[contrast_column] == condition]
    num_cells = group.shape[0]
    X = group.X.toarray() if hasattr(group.X, "toarray") else group.X
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0, ddof=1)
    return {"mu": mu, "sigma": sigma, "num_cells": num_cells}


def compute_s2n(
    adata: AnnData, 
    contrast_column: str, 
    condition_1: str, 
    condition_2: str,
    precomputed_stats: dict | None = None
    
) -> (pd.DataFrame, pd.DataFrame):
    """
    Compute the signal-to-noise ratio (S2N) for each gene between two conditions in an AnnData object.

    Parameters:
    - adata: AnnData object with gene expression data.
    - contrast_column: Key in `adata.obs` indicating the condition labels (e.g. "disease_state").
    - condition_1: Name of the first condition (e.g., "flare").
    - condition_2: Name of the second condition (e.g., "healthy").
    
    Returns:
    - s2n_scores: pandas Series of S2N values indexed by gene names.
    """

    if precomputed_stats is None:
    
        # Select cells for each condition
        group1 = adata[adata.obs[contrast_column] == condition_1]
        group2 = adata[adata.obs[contrast_column] == condition_2]
    
        # print("group1.shape", group1.shape)
        # print("group2.shape", group2.shape)
        
        # number of cells per condition
        num_cells_1 = group1.shape[0]
        num_cells_2 = group2.shape[0]
        
        # Get expression matrices
        X1 = group1.X.toarray() if hasattr(group1.X, "toarray") else group1.X
        X2 = group2.X.toarray() if hasattr(group2.X, "toarray") else group2.X
    
        # Compute mean and std per gene
        mu1 = np.mean(X1, axis=0)
        mu2 = np.mean(X2, axis=0)
        sigma1 = np.std(X1, axis=0, ddof=1)
        sigma2 = np.std(X2, axis=0, ddof=1)

    else:
        vprint("Using precomputed stats", True)
        num_cells_1 = precomputed_stats[condition_1]["num_cells"]
        num_cells_2 = precomputed_stats[condition_2]["num_cells"]
        mu1 = precomputed_stats[condition_1]["mu"]
        mu2 = precomputed_stats[condition_2]["mu"]
        sigma1 = precomputed_stats[condition_1]["sigma"]
        sigma2 = precomputed_stats[condition_2]["sigma"]

    # Compute S2N
    s2n = (mu1 - mu2) / (sigma1 + sigma2 + 1e-8)  # epsilon to avoid division by zero

    num_cells = pd.DataFrame(index=[f"{condition_1}_vs_{condition_2}"], columns=["num_cells_1", "num_cells_2"], data=[[num_cells_1, num_cells_2]])
    stats = pd.DataFrame(s2n, index=adata.var_names, columns=[f"{condition_1}_vs_{condition_2}"])
    
    return stats, num_cells

  
def compute_ranking_statistic(adata: AnnData, contrast_column: str, contrasts: list[Tuple[str, str]]) -> (pd.DataFrame, pd.DataFrame):
    rnk_stats, num_cells = [], []
    precomputed_stats = {}

    conditions = []
    for condition in contrasts:
        conditions.extend([condition[0], condition[1]])
    conditions = np.unique(conditions)
    
    for condition in conditions:
        precomputed_stats[condition] = compute_mu_and_sigma(
            adata, 
            contrast_column=contrast_column, 
            condition=condition
        )

    for condition in contrasts:
        _rnk_stats, _num_cells = compute_s2n(
            adata,
            contrast_column=contrast_column,
            condition_1=condition[0],
            condition_2=condition[1],
            precomputed_stats=precomputed_stats,
        )
        rnk_stats.append(_rnk_stats)
        num_cells.append(_num_cells)
    return pd.concat(rnk_stats, axis=1), pd.concat(num_cells, axis=0)

    
def run_enrichment_test(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype: Tuple[str, str] = ("B cell", "B"),
    direction: str = "upregulated",
    threshold_pval: float = 0.01,
    threshold_lfc: float = 1.,
    threshold_expression: float = 0,
    contrast_column: str = "disease_state",
    celltype_column: str = "disease_state",
    contrasts: Tuple[str, str] | list[Tuple[str, str]] = None,
    min_size: int = 10,
    max_size: int = 1000,
    permutation_num: int = 1000,
    weight: float = 1.,
    max_obs_per_condition: int | None = None, # None = all
    seed: int = 2025,
    verbose: bool = True,
    threads: int = 6,
) -> pd.DataFrame:

    if not isinstance(contrasts, list):
        assert isinstance(contrasts, Tuple)
        contrasts = [contrasts]
        
    celltype_adata = celltype[0]
    celltype_signature = celltype[1]
    
    # filter for cell type
    vprint("Filter for cell type:", verbose)
    adata = adata[adata.obs[celltype_column] == celltype_adata]
    vprint("Filter for cell type: done.", verbose)
    
    # filter based on gene expression
    vprint("Filter for gene expression:", verbose)
    adata = adata[:, adata.X.mean(axis=0) >= threshold_expression]
    vprint("Filter for gene expression: done.", verbose)

    # get genesets
    vprint("Get gene sets:", verbose)
    gene_set_dict, gene_set_df = get_genesets(
        adata=adata,
        df_hcd_all=df_hcd_all,
        celltype_signature=celltype_signature,
        direction=direction,
        threshold_pval=threshold_pval,
        threshold_lfc=threshold_lfc,
    )
    vprint("Get gene sets: done.", verbose)
    
    # compute ranking stat
    vprint("Get ranking stats:", verbose)
    rnk_stats, num_cells_per_condition = compute_ranking_statistic(adata, contrast_column=contrast_column, contrasts=contrasts)
    vprint("Get ranking stats: done.", verbose)
    results = []
    
    for contrast_name in rnk_stats.columns:
        print(contrast_name)
        # format stat so that it can be processed be gseapy
        rnk = rnk_stats.loc[:, contrast_name]\
            .replace([np.inf, -np.inf], np.nan)\
            .dropna()\
            .sort_values(ascending=False)\
            .to_frame()\
            .rename({contrast_name: 1}, axis=1)\
            .rename_axis("0")
        
        # run enrichment
        gp_res = gp.prerank(
            rnk=rnk,
            gene_sets=gene_set_dict,
            min_size=min_size,
            max_size=max_size,
            permutation_num=permutation_num,
            weight=weight,
            outdir=None,
            seed=seed,
            verbose=verbose,
            threads=threads,
        )
        _res = gp_res.res2d
        _res.loc[:, "contrast"] = contrast_name
        _res.loc[:, "num_cells_1"] = num_cells_per_condition.loc[contrast_name, "num_cells_1"]
        _res.loc[:, "num_cells_2"] = num_cells_per_condition.loc[contrast_name, "num_cells_2"]
        _res.loc[:, "percent_duplicate_ranking_stats"] = (rnk.duplicated(keep="first").sum() / rnk.shape[0]) * 100
        results.append(_res)
        vprint(f"{contrast_name}: done.", verbose)

    # combine results and save hyperparams
    results = pd.concat(results, axis=0, ignore_index=True)
    results.loc[:, "celltype_adata"] = celltype_adata
    results.loc[:, "celltype_signature"] = celltype_signature
    results.loc[:, "celltype_combo"] = f"{celltype_adata} ({celltype_signature})"
    results.loc[:, "direction"] = direction
    results.loc[:, "threshold_pval"] = threshold_pval
    results.loc[:, "threshold_lfc"] = threshold_lfc
    results.loc[:, "threshold_expression"] = threshold_expression
    results.loc[:, "min_size"] = min_size
    results.loc[:, "max_size"] = max_size
    results.loc[:, "permutation_num"] = permutation_num
    results.loc[:, "weight"] = weight
    results.loc[:, "seed"] = seed
    results.loc[:, "threads"] = threads

    results.rename({"Term": "cytokine"}, inplace=True, axis=1)
    results = pd.merge(results, gene_set_df, on="cytokine")
    
    return results

In [ ]:
df_hcd_all = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count.csv", index_col=0)
questionable_genes = set(pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/01_overview/questionable_genes.csv")['gene'])
df_hcd_all = df_hcd_all.loc[
    (
        (df_hcd_all['pbs_well'] > 3) 
        & (~df_hcd_all['gene'].isin(questionable_genes))
        & (df_hcd_all['adj_p_value'] < 0.05)
    )
]

In [ ]:
hcd_celltypes = df_hcd_all.celltype.unique()
hcd_celltypes

In [ ]:
adata = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/06_lupus/lupus_norm_log1p.h5ad")

In [ ]:
# df_mart_conversion = pd.read_csv("/Users/soeren.becker/Downloads/mart_export (1).txt", skiprows=0, sep="\t")
# df_mart_conversion.columns = [c.strip().lower().replace(" ", "_") for c in df_mart_conversion.columns]
# id_to_symbol = dict(zip(df_mart_conversion['gene_stable_id'], df_mart_conversion['hgnc_symbol']))
# adata.var['ensembl_id'] = adata.var_names
# adata.var['gene_symbol'] = adata.var_names.map(id_to_symbol)

In [ ]:
ensembl_to_symbol = pd.read_pickle("/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/06_lupus/ens_id_to_symbol.pkl")
adata.var['gene_symbol'] = adata.var_names.map(ensembl_to_symbol)
adata.var.loc[:, "new_index"] = adata.var.gene_symbol
adata.var.loc[adata.var.new_index.isna(), "new_index"] = adata.var.loc[adata.var.new_index.isna()].index
adata.var = adata.var.reset_index().set_index("new_index")
display(adata.var)

display(adata.var_names.value_counts())

adata.var_names_make_unique()

In [ ]:
if False:
    adata_healthy = adata[adata.obs.disease_state == "na"]
    adata_flare = adata[adata.obs.disease_state == "flare"]
    adata_treated = adata[adata.obs.disease_state == "treated"]
    geneset, info_df = get_genesets(adata, df_hcd_all, celltype_signature="NK_CD56low", direction="up", threshold_pval=0.05, threshold_lfc=1.)
    print(adata_healthy[adata_healthy.obs.cell_type == "natural killer cell", geneset["IFN-beta"]].X.mean())
    print(adata_flare[adata_flare.obs.cell_type == "natural killer cell", geneset["IFN-beta"]].X.mean())
    print(adata_treated[adata_treated.obs.cell_type == "natural killer cell", geneset["IFN-beta"]].X.mean())
    df_r = pd.read_csv("./results_robustness_final_DEGs/results_lupus_robustness_[('managed','na'),('flare','na'),('treated','na'),('flare','managed'),('treated','managed'),('flare','treated')]_('naturalkillercell','NK_CD56low')_1_0.01.csv", index_col=0)
    df_r.loc[(df_r.contrast == "flare_vs_treated") & (df_r.cytokine == "IFN-beta")].Lead_genes.values

    lead_geneset = 'IFITM1;ISG15;IFI6;LY6E;MX1;EPSTI1;IFI44L;BST2;IRF7;PRF1;XAF1;IFIT3;RSAD2;CARD16;USP18;OAS2;OAS1;PDGFRB;MX2;CD68;FASLG;TAP2;ANXA5;PTPN6;GALM;FBXO6;TMEM140;RTP4;IFI35;UBE2L6;SAMD9L;PPM1K;NMI;EIF2AK2;ECE1;GBP1;PRR5;NFIL3;STAT1;PARP9;HERC6;KIAA0040;SAMD9;TAP1;OAS3;DDX60L'.split(";")
    print(adata_healthy[adata_healthy.obs.cell_type == "natural killer cell", lead_geneset].X.mean())
    print(adata_flare[adata_flare.obs.cell_type == "natural killer cell", lead_geneset].X.mean())
    print(adata_treated[adata_treated.obs.cell_type == "natural killer cell", lead_geneset].X.mean())

    lead_geneset = 'IFITM1;ISG15;IFI6;LY6E;MX1;EPSTI1;IFI44L;BST2;IRF7;PRF1;XAF1;IFIT3;RSAD2;CARD16;USP18;OAS2;OAS1;PDGFRB;MX2;CD68;FASLG;TAP2;ANXA5;PTPN6;GALM;FBXO6;TMEM140;RTP4;IFI35;UBE2L6;SAMD9L;PPM1K;NMI;EIF2AK2;ECE1;GBP1;PRR5;NFIL3;STAT1;PARP9;HERC6;KIAA0040;SAMD9;TAP1;OAS3;DDX60L'.split(";")
    K=None
    
    for celltype in adata_healthy.obs.cell_type.unique():
        print(celltype)
        print(adata_healthy[adata_healthy.obs.cell_type == celltype, lead_geneset[:K]].X.mean())
        print(adata_flare[adata_flare.obs.cell_type == celltype, lead_geneset[:K]].X.mean())
        print(adata_treated[adata_treated.obs.cell_type == celltype, lead_geneset[:K]].X.mean())
        print("-----")

    for celltype in adata_healthy.obs.cell_type.unique():
        print(celltype)
    
        _genes = geneset["IFN-beta"][np.asarray(adata_flare[adata_flare.obs.cell_type == celltype, geneset["IFN-beta"]].X.mean(axis=0) > 1).squeeze()]
        
        print(adata_flare[adata_flare.obs.cell_type == celltype, _genes].X.mean())
        print(adata_treated[adata_treated.obs.cell_type == celltype, _genes].X.mean())
        
        #print(adata_treated[adata_treated.obs.cell_type == celltype, geneset["IFN-beta"][np.asarray(adata_treated[adata_treated.obs.cell_type == celltype, geneset["IFN-beta"]].X.mean(axis=0) > 1).squeeze()]].X.mean())
        #print(adata_flare[adata_flare.obs.cell_type == celltype, geneset["IFN-beta"][np.asarray(adata_treated[adata_treated.obs.cell_type == celltype, geneset["IFN-beta"]].X.mean(axis=0) > 1).squeeze()]].X.mean())

    for celltype in adata_healthy.obs.cell_type.unique():
        print(celltype)
    
        _genes = geneset["IFN-beta"][np.asarray(adata_flare[adata_flare.obs.cell_type == celltype, geneset["IFN-beta"]].X.mean(axis=0) > 0.1).squeeze()]
        
        print(adata_flare[adata_flare.obs.cell_type == celltype, _genes].X.mean())
        print(adata_treated[adata_treated.obs.cell_type == celltype, _genes].X.mean())
        
        #print(adata_treated[adata_treated.obs.cell_type == celltype, geneset["IFN-beta"][np.asarray(adata_treated[adata_treated.obs.cell_type == celltype, geneset["IFN-beta"]].X.mean(axis=0) > 1).squeeze()]].X.mean())
        #print(adata_flare[adata_flare.obs.cell_type == celltype, geneset["IFN-beta"][np.asarray(adata_treated[adata_treated.obs.cell_type == celltype, geneset["IFN-beta"]].X.mean(axis=0) > 1).squeeze()]].X.mean())

    for celltype in adata.obs.cell_type.unique():
        key_genes = ["IFITM1", "IFIT1", "IFIT2", "IFIT3"]
        sc.pl.violin(
            adata[adata.obs.cell_type == celltype, key_genes],
            keys=key_genes, 
            groupby="disease_state", 
            rotation=90,
            order=["na", "managed", "flare", "treated"],
            stripplot=False,
            show=False
        )
        plt.suptitle(celltype)
        plt.show()

In [ ]:
celltype_combos = (
    ("B cell", "B_cell"),
    ('CD4-positive, alpha-beta T cell', "CD4_T_cell"),
    ('CD8-positive, alpha-beta T cell', "CD8_T_cell"),
    ('classical monocyte', "CD14_Mono"),
    ('non-classical monocyte', "CD16_Mono"),
    ('natural killer cell', "NK_CD56hi"),
    ('natural killer cell', "NK_CD56low"),
    ('conventional dendritic cell', "cDC"),
)

for combo in celltype_combos:
    assert combo[1] in hcd_celltypes

In [ ]:
if False:
    contrasts = itertools.product(adata.obs.disease_state.unique(), adata.obs.disease_state.unique())
    contrasts = [c for c in contrasts if c[0] != c[1]]
    contrasts

contrasts = [
    ('managed', 'na'), ('flare', 'na'), ('treated', 'na'), 
    ("flare", "managed"), ("treated", "managed"), 
    ("flare", "treated"),
]

In [ ]:
adata.obs.cell_type.value_counts()

In [ ]:
if False:
    # saving per cell type to avoid OOM errors

    for celltype in adata.obs.cell_type.unique():
        print(f"Saving {celltype}")
        adata[adata.obs.cell_type == celltype].write_h5ad(
            f"/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/06_lupus/lupus_norm_log1p_{celltype}.h5ad"
        )
    del adata

In [ ]:
os.makedirs("./results_robustness_final_DEGs", exist_ok=True)

In [ ]:
threshold_pval = 0.05

for celltype_combo_k, celltype_combo in enumerate(celltype_combos):
    # all_results = []
    print(celltype_combo)
    shared_kwargs = {
        "threshold_pval": threshold_pval,
        "direction": "upregulated",
        "min_size": 8,
        "max_size": 1_000,
        "permutation_num": 20_000,
        "verbose": True,
        "df_hcd_all": df_hcd_all,
        "adata": sc.read_h5ad(
            f"/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/06_lupus/lupus_norm_log1p_{celltype_combo[0]}.h5ad"
        )
    }

    for threshold_lfc in [0.8, 0.9, 1, 1.1, 1.2,]:

        shared_kwargs["threshold_lfc"] = threshold_lfc
        
        for threshold_expression in [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:
            
            shared_kwargs["threshold_expression"] = threshold_expression
            
            results = run_enrichment_test(
                celltype=celltype_combo,
                contrast_column="disease_state",
                celltype_column="cell_type",
                contrasts=contrasts,
                **shared_kwargs,
            )
            
            _filename = f"results_lupus_robustness_{contrasts}_{celltype_combo}_{threshold_lfc}_{threshold_expression}.csv".replace(" ", "")
            _filename = os.path.join("./results_robustness_final_DEGs", _filename)
            vprint(f"Saving at: {_filename}", True)
            results.to_csv(_filename)
            # all_results.append(results)

    # all_results = pd.concat(all_results, axis=0)